Day 5, Topic 1: Memory Layout & CPU Cache – C‑order vs. F‑order Deep Dive

In [2]:
import numpy as np

arr_2d_c = np.arange(12, dtype=np.int32).reshape(3, 4, order='C') 
arr_2d_f = np.arange(12, dtype=np.int32).reshape(3, 4, order='F')

print("Strides of C-order matrix: ", arr_2d_c.strides)
print("Strides of F-order matrix: ", arr_2d_f.strides)

Strides of C-order matrix:  (16, 4)
Strides of F-order matrix:  (4, 12)


In [10]:
import time

size = 8000
arr_c = np.ones((size, size), dtype=np.float64, order='C')
arr_f = np.ones((size, size), dtype=np.float64, order='F')

# Sum over rows (axis=1) – good for C-order
start = time.perf_counter()
sum_rows_c = arr_c.sum(axis=1)
time_rows_c = time.perf_counter() - start

# Sum over columns (axis=0) – bad for C-order
start = time.perf_counter()
sum_cols_c = arr_c.sum(axis=0)
time_cols_c = time.perf_counter() - start

print(f"C-order: sum rows (axis=1): {time_rows_c:.4f} s")
print(f"C-order: sum cols (axis=0): {time_cols_c:.4f} s")
print(f"C-order slowdown factor: {time_cols_c / time_rows_c:.2f}x")

C-order: sum rows (axis=1): 0.0849 s
C-order: sum cols (axis=0): 0.0476 s
C-order slowdown factor: 0.56x


In [11]:
start = time.perf_counter()
sum_rows_f = arr_f.sum(axis=1)
time_rows_f = time.perf_counter() - start

start = time.perf_counter()
sum_cols_f = arr_f.sum(axis=0)
time_cols_f = time.perf_counter() - start

print(f"F-order: sum rows (axis=1): {time_rows_f:.4f} s")
print(f"F-order: sum cols (axis=0): {time_cols_f:.4f} s")
print(f"F-order slowdown factor (rows/cols): {time_rows_f / time_cols_f:.2f}x")

F-order: sum rows (axis=1): 0.0737 s
F-order: sum cols (axis=0): 0.1097 s
F-order slowdown factor (rows/cols): 0.67x


In [12]:
arr_3d_c = np.ones((100, 200, 300), dtype=np.float64, order='C')
arr_3d_f = np.ones((100, 200, 300), dtype=np.float64, order='F')

#c-order timings
%timeit arr_3d_c.sum(axis=2)
%timeit arr_3d_c.sum(axis=0)

#f-order timings
%timeit arr_3d_f.sum(axis=0)
%timeit arr_3d_f.sum(axis=2)

6.5 ms ± 130 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
5.04 ms ± 56.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.39 ms ± 64.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.48 ms ± 22.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Practice Activity: Memory Layout Performance

In [13]:
#Create two large square matrices A and B of size 2000×2000 with dtype=float64, both in C‑order.
size = 2000
A = np.ones((size, size), dtype=np.float64, order='C')
B = np.ones((size, size), dtype=np.float64, order='C')

In [17]:
#Time the matrix multiplication A @ B using np.matmul or the @ operator.
%timeit A @ B

149 ms ± 35.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
#Now convert B to F‑order using np.asfortranarray(B) and time the multiplication again.
%timeit A @ np.asfortranarray(B)

240 ms ± 30.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
#Explain why one is faster than the other. Ans: C order is faster

In [21]:
# Use %timeit to compare np.dot(A, B) with np.dot(A, B.T) when both are C‑order. 
#Why is one significantly faster?
%timeit np.dot(A, B)
%timeit np.dot(A, B.T)

147 ms ± 25.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
150 ms ± 14.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Day 5, Topic 2: The Temporary Array Trap – Hidden Allocations That Kill Performance

In [22]:
import numpy as np
a = np.ones(10)
b = np.ones(10)
c = np.ones(10)
d = np.ones(10)

np.multiply(a, b, out=a)
np.multiply(c, d, out=c)
a += c

In [23]:
a = np.ones(10_000_000)
b = np.ones(10_000_000)
c = np.ones(10_000_000)
d = np.ones(10_000_000)

result = np.empty_like(a)

temp = np.empty_like(a)
np.multiply(a, b, out=temp)

np.multiply(c, d, out=a)

np.add(temp, a, out=result)

array([2., 2., 2., ..., 2., 2., 2.], shape=(10000000,))

In [24]:
#a*b + c*d + e*f + g*h
a = np.ones(10_000_000)
b = np.ones(10_000_000)
c = np.ones(10_000_000)
d = np.ones(10_000_000)
e = np.ones(10_000_000)
f = np.ones(10_000_000)
g = np.ones(10_000_000)
h = np.ones(10_000_000)

temp = np.empty_like(a)
result = np.empty_like(a)

#a*b
np.multiply(a, b, out=result)

#Add c*d
np.multiply(c, d, out=temp)
np.add(temp, result, out=result)

#Add e*f
np.multiply(e, f, out=temp)
np.add(result, temp, out=result)

#Add g*h
np.multiply(g, h, out=temp)
np.add(result, temp, out=result)


array([4., 4., 4., ..., 4., 4., 4.], shape=(10000000,))

In [25]:
a = np.ones((3, 1))
b = np.ones((1, 4))
result = np.empty((3, 4))
np.add(a, b, out=result)

array([[2., 2., 2., 2.],
       [2., 2., 2., 2.],
       [2., 2., 2., 2.]])

In [28]:
images = np.random.randint(0, 256, size=(100, 256, 256, 3), dtype=np.uint8)
batch_size = 100

buffer = np.empty_like(images, dtype=np.float32)

np.divide(images, 255.0, out=buffer)

np.power(buffer, 0.5, out=buffer)

means = buffer.mean(axis=(1, 2, 3), keepdims=True)
np.subtract(buffer, means, out=buffer).shape

(100, 256, 256, 3)

Practice Activity: The Temporary Array Trap

In [29]:
#result = (x + y) * (y - z) + np.sqrt(x * z)

In [38]:
import numpy as np

# Create large arrays
size = 20_000_000
x = np.random.rand(size).astype(np.float32)  # 80 MB each (float32)
y = np.random.rand(size).astype(np.float32)
z = np.random.rand(size).astype(np.float32)

# ----- Naive Version -----
def naive(x, y, z):
    return (x + y) * (y - z) + np.sqrt(x * z)

# ----- Optimized Version -----
def optimized(x, y, z):
    temp = np.empty_like(x)
    result = np.empty_like(x)
    np.add(x, y, out=result)
    np.subtract(y, z, out=temp)
    np.multiply(result, temp, out=result)
    np.multiply(x, z, out=temp)
    np.sqrt(temp, out=temp)
    np.add(result, temp, out=result)
    return result

# Verify correctness
np.testing.assert_allclose(naive(x, y, z), optimized(x, y, z), rtol=1e-5)

# Time both
print("Naive version:")
%timeit naive(x, y, z)

print("\nOptimized version:")
%timeit optimized(x, y, z)

Naive version:
244 ms ± 82.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Optimized version:
135 ms ± 4.81 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Day 5, Topic 3: Vectorization vs. Python Loops – When Loops Actually Win

In [39]:
import numpy as np
import time

size = 10
a = np.random.randn(size)
b = np.random.randn(size)

def python_dot(a, b):
    total = 0.0
    for i in range(len(a)):
        total += a[i] + b[i]
    return total

%timeit python_dot(a, b)
%timeit np.dot(a, b)

4.48 μs ± 239 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
1.57 μs ± 31.7 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [4]:
size = 50_000_000
a = np.random.randn(size).astype(np.float32)
b = np.random.randn(size).astype(np.float32)
c = np.random.randn(size).astype(np.float32)
d = np.random.randn(size).astype(np.float32)

def vectorized(a, b, c, d):
    return (a + b) * (c - d) + np.sqrt(a * c)

def looped(a, b, c, d):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = (a[i] + b[i]) * (c[i] - d[i]) + np.sqrt(a[i] * c[i])
    return out

%timeit vectorized(a, b, c, d)
%timeit looped(a, b, c, d)

C:\Users\Faizan Toheed\AppData\Local\Temp\ipykernel_2484\946777610.py:8: RuntimeWarning: invalid value encountered in sqrt
  return (a + b) * (c - d) + np.sqrt(a * c)


796 ms ± 64.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


C:\Users\Faizan Toheed\AppData\Local\Temp\ipykernel_2484\946777610.py:13: RuntimeWarning: invalid value encountered in sqrt
  out[i] = (a[i] + b[i]) * (c[i] - d[i]) + np.sqrt(a[i] * c[i])


KeyboardInterrupt: 

In [3]:
arr = np.random.randn(1000)

def python_first_neg(arr):
    for i, v in enumerate(arr):
        if v < 0:
            return i
    
    return -1

def numpy_first_neg(arr):
    neg = np.where(arr < 0)[0] 
    return neg[0] if len(neg) > 0 else -1

%timeit python_first_neg(arr)
%timeit numpy_first_neg(arr)

578 ns ± 49.1 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
6.31 μs ± 492 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [5]:
from numba import njit

@njit
def numba_looped(a, b, c, d):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = (a[i] + b[i]) * (c[i] - d[i]) + np.sqrt(a[i] * c[i])
    return out

%timeit numba_looped(a, b, c, d)

172 ms ± 15.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
import numpy as np

def python_loop_correct(a, b):
    out = np.empty_like(a)
    for i in range(len(a)):
        out[i] = a[i] * (b[i] + a[i])
    return out

def vectorized(a, b):
    return a * (a + b)

# Test for various N
for N in [10, 100, 1000, 10000, 100000]:
    a = np.random.randn(N).astype(np.float32)
    b = np.random.randn(N).astype(np.float32)
    
    print(f"\nN = {N}")
    print("Python loop:")
    %timeit python_loop_correct(a, b)
    print("Vectorized:")
    %timeit vectorized(a, b)


N = 10
Python loop:
7.85 μs ± 550 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
Vectorized:
1.74 μs ± 92.4 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)

N = 100
Python loop:
65.2 μs ± 8.78 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
Vectorized:
1.97 μs ± 223 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

N = 1000
Python loop:
756 μs ± 95.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Vectorized:
3.56 μs ± 602 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

N = 10000
Python loop:
7.56 ms ± 1.55 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
Vectorized:
10 μs ± 1.45 μs per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

N = 100000
Python loop:
84.5 ms ± 8.47 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
Vectorized:
85.9 μs ± 4.03 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
